In [3]:
import ROOT
import math 
import time
import os
import numpy as np
from array import array
import uproot
import pandas as pd

In [4]:
# Get input TTree
#dir_base = '/eos/home-l/lportale/VBFparkingNtuples/merged/'
dir_base = ''
dir_in = dir_base
DFs = {}

file_in = {
    #"24H" : "/eos/home-l/lportale/VBFparkingNtuples/histos_24H.root",
    #"25BC" : "/eos/home-l/lportale/VBFparkingNtuples/250610_1b1tau_fixL1/histos_25BC.root",
    #"25D" : "/eos/home-l/lportale/VBFparkingNtuples/histos_25D_250729.root"
    #"25CD" : "/eos/user/l/lportale/VBFparkingNtuples/histos_Run25CD_250730.root",
    "25CD": "/eos/user/l/lportale/VBFparkingNtuples/histos_25CD_renano.root",
    #"24all" : "/eos/user/l/lportale/VBFparkingNtuples/histos_Run24all_250730.root",

}
df = {}
for key in file_in:
    #df[key] = ROOT.RDataFrame('VBFNanoAOD/VBFtree',file_in[key])
    df[key] = ROOT.RDataFrame('VBFtree',file_in[key])

In [5]:

### sort objects by X (e.g. btag score)
ROOT.gInterpreter.Declare(

'''
#include <iostream>
#include <vector>
#include <numeric>      // std::iota
#include <algorithm>    // std::sort, std::stable_sort

using namespace std;

ROOT::VecOps::RVec<Float_t> sort_indexes(ROOT::VecOps::RVec<Float_t> &v) {

  // initialize original index locations
  ROOT::VecOps::RVec<Float_t> idx(v.size());
  iota(idx.begin(), idx.end(), 0);

  stable_sort(idx.begin(), 
              idx.end(),
              [&v](size_t i1, size_t i2) {return v[i1] < v[i2];});

  return idx;
}
''')

### Evaluate HT
ROOT.gInterpreter.Declare(

'''
Float_t get_HT(ROOT::VecOps::RVec<Float_t> pt, ROOT::VecOps::RVec<Float_t> eta, ROOT::VecOps::RVec<Float_t> id, int idcut){
    const auto size = pt.size();
    float HT=0.;
    for (size_t idx1=0; idx1<size; ++idx1){
        if (pt[idx1]>20. && fabs(eta[idx1])<2.5 && id[idx1]>=idcut)
            HT+=pt[idx1];
    }
    return HT;
}

'''
)


### Evaluate number of central jets
ROOT.gInterpreter.Declare(
'''
int get_ncentraljets(ROOT::VecOps::RVec<Float_t> pt, 
                     ROOT::VecOps::RVec<Float_t> eta,
              ROOT::VecOps::RVec<Float_t> id, int idcut)
{
    const auto size = pt.size();
    int njet=0;
    for (size_t idx1=0; idx1<size; ++idx1){
        if(pt[idx1]>30 && fabs(eta[idx1])<2.5 and id[idx1]>=idcut) njet++;
    }
    return njet;
}
'''
)


### Evaluate number of central jets
ROOT.gInterpreter.Declare(
'''
Float_t get_PNetBTagMean(ROOT::VecOps::RVec<Float_t> pt, 
                     ROOT::VecOps::RVec<Float_t> eta,
                     ROOT::VecOps::RVec<Float_t> id, int idcut,
                     ROOT::VecOps::RVec<Float_t> score)
{
    const auto size = pt.size();
    Float_t best_score = -1;
    Float_t sec_score = -1;
    for (size_t idx1=0; idx1<size; ++idx1){
        if(pt[idx1]>30 && fabs(eta[idx1])<2.5 and id[idx1]>=idcut)
        {
         if (score[idx1] > best_score) {
             sec_score = best_score;
             best_score = score[idx1];
         }
         else if (score[idx1] > sec_score) sec_score = score[idx1];
        }
    }
    return 0.5*(best_score+sec_score);
}
'''
)

True

In [6]:
def preped_df_HH(df, period):
    idcut=1
    if "24" in period:
        idcut=6
    df_tmp = df
    df_tmp = df_tmp.Define('njets_cent','get_ncentraljets(jets_pt,jets_eta,jets_id,{})'.format(idcut))\
                   .Filter('njets_cent>3')\
                   .Define('bTag_idx','sort_indexes(jets_btagPNetB)')\
                   .Define('HT','get_HT(jets_pt,jets_eta,jets_id,{})'.format(idcut))\
                   .Define('sublead_bscore','bTag_idx[1]')\
                   .Define('lead_bscore','bTag_idx[0]')\
                   .Define('PNetMean','get_PNetBTagMean(jets_pt,jets_eta,jets_id,{},jets_btagPNetB)'.format(idcut))
    return df_tmp

def preped_df_HH1(df):
    df_tmp = df
    df_tmp = df_tmp.Define('njets_cent','get_ncentraljets(jets_pt,jets_eta)')\
                   .Filter('njets_cent>3')\
                   .Define('sublead_bscore','bTag_idx[1]')\
                   .Define('lead_bscore','bTag_idx[0]')\
                   .Define('PNetMean','0.5*(jets_btagPNetB[sublead_bscore]+jets_btagPNetB[lead_bscore])')
    return df_tmp

In [8]:
df_HH = {}
for key in file_in:
    df_HH[key] = preped_df_HH(df[key],"25")

In [10]:
### RUN THIS ONLY ONCE, IT TAKES A WHILE
for key in file_in:
    df_HH[key].Snapshot("VBFNanoAOD", "/eos/home-l/lportale/VBFparkingNtuples/snapshots/histos_ParkingHH_250808_{}.snapshot.root".format(key))

In [11]:
file_in = {
    #"24H" : "/eos/home-l/lportale/VBFparkingNtuples/histos_24H.root",
    #"25BC" : "/eos/home-l/lportale/VBFparkingNtuples/250610_1b1tau_fixL1/histos_25BC.root",
    #"25D" : "/eos/home-l/lportale/VBFparkingNtuples/histos_25D_250729.root"
    #"25CD" : "/eos/user/l/lportale/VBFparkingNtuples/histos_Run25CD_250730.root",
    "25CD": "/eos/user/l/lportale/VBFparkingNtuples/histos_25CD_renano.root",
    "24all" : "/eos/user/l/lportale/VBFparkingNtuples/histos_Run24all_250730.root",

}

In [29]:
### ONCE IT RAN, START FROM THIS INSTEAD
df_HH = {}

for key in file_in:
    idcut=1
    if "24" in key:
        idcut=6
        df_HH[key] = ROOT.RDataFrame('VBFNanoAOD',"/eos/home-l/lportale/VBFparkingNtuples/snapshots/histos_ParkingHH_250730_{}.snapshot.root".format(key))\
                       .Redefine('njets_cent','get_ncentraljets(jets_pt,jets_eta,jets_id,{})'.format(idcut))\
                       .Filter('njets_cent>3')\
                       .Redefine('bTag_idx','sort_indexes(jets_btagPNetB)')\
                       .Redefine('HT','get_HT(jets_pt,jets_eta,jets_id,{})'.format(idcut))\
                       .Redefine('sublead_bscore','bTag_idx[1]')\
                       .Redefine('lead_bscore','bTag_idx[0]')\
                       .Redefine('PNetMean','get_PNetBTagMean(jets_pt,jets_eta,jets_id,{},jets_btagPNetB)'.format(idcut))

    else:
        df_HH[key] = ROOT.RDataFrame('VBFNanoAOD',"/eos/home-l/lportale/VBFparkingNtuples/snapshots/histos_ParkingHH_250808_{}.snapshot.root".format(key))\
                       .Redefine('njets_cent','get_ncentraljets(jets_pt,jets_eta,jets_id,{})'.format(idcut))\
                       .Filter('njets_cent>3')\
                       .Redefine('bTag_idx','sort_indexes(jets_btagPNetB)')\
                       .Redefine('HT','get_HT(jets_pt,jets_eta,jets_id,{})'.format(idcut))\
                       .Redefine('sublead_bscore','bTag_idx[1]')\
                       .Redefine('lead_bscore','bTag_idx[0]')\
                       .Redefine('PNetMean','get_PNetBTagMean(jets_pt,jets_eta,jets_id,{},jets_btagPNetB)'.format(idcut))



In [30]:
#print(df_HH['24H'].Count().GetValue())
#print(df_HH['25BC'].Count().GetValue())
#print(df_HH['25D'].Count().GetValue())
print(df_HH['25CD'].Count().GetValue())

912354


In [31]:
df_HH_num = {}
for key in file_in:
    df_HH_num[key] = df_HH[key].Filter('passHHparking')

In [32]:
#print(df_HH_num['24H'].Count().GetValue())
#print(df_HH_num['25BC'].Count().GetValue())
#print(df_HH_num['25D'].Count().GetValue())
print(df_HH_num['25CD'].Count().GetValue())

148773


In [33]:
df_HH_num["25C"] = df_HH_num['25CD'].Filter('run<394299')
df_HH_num["25D"] = df_HH_num['25CD'].Filter('run>=394299')

df_HH["25C"] = df_HH['25CD'].Filter('run<394299')
df_HH["25D"] = df_HH['25CD'].Filter('run>=394299')


In [22]:
### 2D HT vs 2btagMean

h_num = {}
h_den = {}
h_eff = {}
for key in df_HH_num:
    h_num[key] = df_HH_num[key].Histo2D(("hnum_{}".format(key),"HT vs PNet - {}".format(key),20,200,1000,20,0,1),"HT","PNetMean")
    h_den[key] = df_HH[key].Histo2D(("hden_{}".format(key),"HT vs PNet - {}".format(key),20,200,1000,20,0,1),"HT","PNetMean")
    #h_eff[key] = h_num[key].Divide(h_den[key].GetPtr())
    
    h_num_ptr = h_num[key].GetPtr()
    h_den_ptr = h_den[key].GetPtr()
    h_eff[key] = h_num_ptr.Clone(f"h_eff_{key}")
    h_eff[key].Divide(h_num_ptr, h_den_ptr, 1.0, 1.0, "B")  # "B" → binomial errors

In [23]:
import CMS_lumi, tdrstyle
import array

tdrstyle.setTDRStyle()

#change the CMS_lumi variables (see CMS_lumi.py)
CMS_lumi.writeExtraText = True
CMS_lumi.extraText = "  Preliminary"
#CMS_lumi.lumi_sqrtS = "22.1 fb^{-1}, 2023 (13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)
CMS_lumi.lumi_sqrtS = "2024(C+D+E+F) (13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)

iPeriod = 0
iPos = 0#11
H_ref = 600; 
W_ref = 600; 
W = W_ref
H = H_ref
# references for T, B, L, R
T = 0.08*H_ref
B = 0.14*H_ref 
L = 0.12*W_ref
R = 0.04*W_ref


In [24]:
import ROOT
import CMS_lumi  # make sure this is available

c = ROOT.TCanvas("c12311", "c", 800, 700)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPaintTextFormat("4.2f")

for key, h in h_eff.items():
    c.Clear()
    c.SetFillColor(0)
    c.SetBorderMode(0)
    c.SetFrameFillStyle(0)
    c.SetFrameBorderMode(0)
    
    # Correct margins
    c.SetRightMargin(0.20)   # reserve space for palette
    c.SetLeftMargin(0.13)
    c.SetTopMargin(0.08)
    c.SetBottomMargin(0.12)

    # Draw histogram
    h.SetTitle("")
    h.GetXaxis().SetTitle("HT [GeV]")
    h.GetYaxis().SetTitle("PNet2BTagMean")
    h.GetZaxis().SetTitle("Trigger Efficiency")
    h.GetZaxis().SetRangeUser(0, 1)
    h.Draw("COLZ")

    # Move the color palette manually
    palette = h.GetListOfFunctions().FindObject("palette")
    if palette:
        palette.SetX1NDC(0.86)
        palette.SetX2NDC(0.91)

    # --- CMS Label with CMS_lumi ---
    CMS_lumi.lumi_sqrtS = "20{} (13.6 TeV)".format(key)
    CMS_lumi.CMS_lumi(c, iPeriod, iPos)

    # --- Extra annotations ---
    latex = ROOT.TLatex()
    latex.SetNDC()
    latex.SetTextFont(42)
    latex.SetTextSize(0.035)
    latex.DrawLatex(0.15, 0.75, "Dataset: Muon")
    latex.DrawLatex(0.15, 0.70, "Reference HLT:")
    latex.DrawLatex(0.15, 0.66, "isolated #mu, p_{T} > 24 GeV")
    latex.DrawLatex(0.15, 0.60, "Event selection:")
    latex.DrawLatex(0.15, 0.56, "#geq 4 jets, p_{T} > 30 GeV, |#eta| < 2.5")

    # Save output
    outname = f"/eos/home-l/lportale/VBFparkingNtuples/ParkingHH_2B_eff2D_{key}"
    c.SaveAs(outname + ".png")
    c.SaveAs(outname + ".pdf")


Info in <TCanvas::Print>: png file /eos/home-l/lportale/VBFparkingNtuples/ParkingHH_2B_eff2D_25CD.png has been created
Info in <TCanvas::Print>: pdf file /eos/home-l/lportale/VBFparkingNtuples/ParkingHH_2B_eff2D_25CD.pdf has been created
Info in <TCanvas::Print>: png file /eos/home-l/lportale/VBFparkingNtuples/ParkingHH_2B_eff2D_24all.png has been created
Info in <TCanvas::Print>: pdf file /eos/home-l/lportale/VBFparkingNtuples/ParkingHH_2B_eff2D_24all.pdf has been created
Info in <TCanvas::Print>: png file /eos/home-l/lportale/VBFparkingNtuples/ParkingHH_2B_eff2D_25C.png has been created
Info in <TCanvas::Print>: pdf file /eos/home-l/lportale/VBFparkingNtuples/ParkingHH_2B_eff2D_25C.pdf has been created
Info in <TCanvas::Print>: png file /eos/home-l/lportale/VBFparkingNtuples/ParkingHH_2B_eff2D_25D.png has been created
Info in <TCanvas::Print>: pdf file /eos/home-l/lportale/VBFparkingNtuples/ParkingHH_2B_eff2D_25D.pdf has been created


In [25]:
def fill_eff(h_num, h_den, year,var):
    eff = ROOT.TGraphAsymmErrors(h_num[year].GetValue(),
                                 h_den[year].GetValue(),
                                 'eff_{}'.format(var))
    return eff
 
def redrawBorder():
    ROOT.gPad.Update()
    ROOT.gPad.RedrawAxis()
    l = ROOT.TLine()
    l.SetLineWidth(2)
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymax(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymin())


In [26]:
### 1D HT
h_num = {}
h_den = {}
Eff = {}
for key in df_HH:
    h_num[key] = df_HH_num[key].Filter("PNetMean>0.6").Histo1D(("hnum_{}".format(key),"HT - {}".format(key),10,200,1000),"HT")
    h_den[key] = df_HH[key].Filter("PNetMean>0.6").Histo1D(("hden_{}".format(key),"HT - {}".format(key),10,200,1000),"HT")


    Eff[key] = fill_eff(h_num, h_den, key, 'HT')

In [27]:
# Defining variables, categories & cuts

jet_vars = ['HT','pt1','pt2','mjj','detajj','chf']

add_cut_label = {
    'HT'    : ' ',
}

# Plotting labels
VARLABELS = {
    #'pnet'   :  ' Transformed PNet #tau-tag score ',
    'HT'      : 'H_{T} [GeV]',
    'pnetraw'   :  ' PNet #tau-tag score (Vs Jet) '
}

import CMS_lumi, tdrstyle
import array

tdrstyle.setTDRStyle()

#change the CMS_lumi variables (see CMS_lumi.py)
CMS_lumi.writeExtraText = True
CMS_lumi.extraText = "  Preliminary"
#CMS_lumi.lumi_sqrtS = "22.1 fb^{-1}, 2023 (13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)
CMS_lumi.lumi_sqrtS = "2024(C+D+E+F) (13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)

iPeriod = 0
iPos = 0#11
H_ref = 600; 
W_ref = 600; 
W = W_ref
H = H_ref
# references for T, B, L, R
T = 0.08*H_ref
B = 0.14*H_ref 
L = 0.12*W_ref
R = 0.04*W_ref



In [28]:
### #######################################

CMS_lumi.lumi_sqrtS = "13.6 TeV" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)
theVar = 'HT'

path = {}
canvas = {}
hpx = {}
legend = {}

text_coord = {
    'HT':    [0.4,0.3,2.9,0.8],
}


Eff['24all'].SetLineColor(46)
Eff['24all'].SetMarkerColor(46)
Eff['24all'].SetMarkerSize(2)
Eff['24all'].SetLineWidth(3)
Eff['24all'].SetMarkerStyle(23)   


Eff['25C'].SetLineColor(8)
Eff['25C'].SetMarkerColor(8)
Eff['25C'].SetMarkerSize(2)
Eff['25C'].SetLineWidth(3)
Eff['25C'].SetMarkerStyle(23)   


Eff['25D'].SetLineColor(6)
Eff['25D'].SetMarkerColor(6)
Eff['25D'].SetMarkerSize(2)
Eff['25D'].SetLineWidth(3)
Eff['25D'].SetMarkerStyle(23)   

path['25C'] = ROOT.TPaveText(text_coord[theVar][0],
                              text_coord[theVar][1],
                              text_coord[theVar][2],
                              text_coord[theVar][3])

path['25C'].SetTextAlign(12)
path['25C'].SetFillStyle(0)
path['25C'].SetBorderSize(0)

path['25C'].AddText('Offline selection: ')
path['25C'].AddText('1 #mu (tight), p_{T}> 26 GeV')
path['25C'].AddText('#geq 4 jets, p_{T} > 30 GeV, |#eta|<2.5')
path['25C'].AddText('PNet2BTagMean > 0.5')

c=ROOT.TCanvas('bazeaaaaab',"Trigger Efficicency",1280,1024)
c.SetFillColor(0)
c.SetBorderMode(0)
c.SetFrameFillStyle(0)
c.SetFrameBorderMode(0)
c.SetLeftMargin( L/W )
c.SetRightMargin( R/W )
c.SetTopMargin( T/H )
c.SetBottomMargin( B/H )
c.SetTickx()
c.SetTicky()
c.cd()

#dummy hist for consistent display
xlow=Eff['25C'].GetXaxis().GetBinLowEdge(0)
xhigh=Eff['25C'].GetXaxis().GetBinUpEdge(92)
hpx['25C'] = ROOT.TH2F("hpx","",10,xlow,xhigh,10,-0.02,1.2)
hpx['25C'].SetStats(False)
hpx['25C'].SetTitle(theVar)
hpx['25C'].GetXaxis().SetTitle(VARLABELS[theVar])
hpx['25C'].GetYaxis().SetTitle("Trigger Efficiency")
hpx['25C'].GetXaxis().SetTitleSize(0.055)
hpx['25C'].GetXaxis().SetTitleOffset(1.1)
hpx['25C'].GetYaxis().SetTitleSize(0.06)
hpx['25C'].GetYaxis().SetTitleOffset(0.9)
hpx['25C'].Draw()

legend['25C'] = ROOT.TLegend(0.15,0.73,0.5,0.88) # top left
legend['25C'].SetTextSize(0.04)
legend['25C'].SetFillStyle(0)
legend['25C'].SetBorderSize(0)
#legend['25BC'].AddEntry(Eff['B'][theVar]['new'],     'Era B',                 'lp')
#legend['25BC'].AddEntry(Eff['C'][theVar]['new'],     'Era C',                 'lp')
#legend['25BC'].AddEntry(Eff['25BC'][theVar]['new'],     'Era D',                 'lp')
#legend['25BC'].AddEntry(Eff['E'][theVar]['new'], 'Era E', 'lp')
legend['25C'].AddEntry(Eff['24all'],   '2024',      'lp')
legend['25C'].AddEntry(Eff['25C'], '2025 C',      'lp')
legend['25C'].AddEntry(Eff['25D'],   '2025 D',      'lp')

#path['25BC'].SetTextSize(0.045);

#Eff['B'][theVar]['new'].Draw("p same")
#Eff['C'][theVar]['new'].Draw("p same")
#Eff['25BC'][theVar]['new'].Draw("p same")
#Eff['25BC'][theVar]['new'].Draw("p same")
#Eff['E'][theVar]['new'].Draw("p same")
#Eff['25BC'].Draw("p same")
Eff['24all'].Draw("p same")
Eff['25C'].Draw("p same")
Eff['25D'].Draw("p same")

legend['25C'].Draw("same")
path['25C'].Draw("same")
c.Update()
CMS_lumi.CMS_lumi(c, iPeriod, iPos)


# --- Extra annotations ---
latex = ROOT.TLatex()
latex.SetNDC()
latex.SetTextFont(42)
latex.SetTextSize(0.035)

offset_x=0.35
offset_y=-0.3
latex.DrawLatex(0.15+offset_x, 0.75+offset_y, "Dataset: Muon")
latex.DrawLatex(0.15+offset_x, 0.70+offset_y, "Reference HLT:")
latex.DrawLatex(0.15+offset_x, 0.66+offset_y, "isolated #mu, p_{T} > 24 GeV")
latex.DrawLatex(0.15+offset_x, 0.60+offset_y, "Event selection:")
latex.DrawLatex(0.15+offset_x, 0.56+offset_y, "#geq 4 jets, p_{T} > 30 GeV, |#eta| < 2.5")
latex.DrawLatex(0.15+offset_x, 0.52+offset_y, "PNet2BTagMean > 0.6")

redrawBorder()

#canvas[theVar].Draw()
c.Draw()


c.SaveAs('/eos/home-l/lportale/testcanvas.png')
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.pdf".format(theVar))
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.png".format(theVar))
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.root".format(theVar))
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.C".format(theVar))


Info in <TCanvas::Print>: png file /eos/home-l/lportale/testcanvas.png has been created


In [ ]:
### MET trigger plots

df_den = {}#df_mjj_all

# define some filters and variables
#pass_cuts_for_mjj = 'METnoMu>200. && pass_VBF_cuts("mjj",jets_pt,jets_eta,jets_phi,jets_id,jets_CHF,115.,50.,800,3.5, false)'
#pass_cuts_for_detajj = 'METnoMu>200. && pass_VBF_cuts("detajj",jets_pt,jets_eta,jets_phi,jets_id,jets_CHF,115.,50.,800,3.5, false)'
#pass_cuts_for_MET = 'mjj2>800. && pass_VBF_cuts("mjj",jets_pt,jets_eta,jets_phi,jets_id,jets_CHF,115.,50.,800,3.5, false)'

#pass_cuts_for_mjj = 'pass_VBF_cuts("mjj",jets_pt,jets_eta,jets_phi,jets_id,jets_CHF,125.,55.,1100,4.0, false)'
#pass_cuts_for_detajj = 'pass_VBF_cuts("detajj",jets_pt,jets_eta,jets_phi,jets_id,jets_CHF,125.,55.,1100,4.0, false)'
#pass_cuts_for_MET = 'mjj2>800. && pass_VBF_cuts("mjj",jets_pt,jets_eta,jets_phi,jets_id,jets_CHF,115.,50.,800,3.5, false)'


pass_cuts_for_mjj = 'pass_VBF_cuts("mjj",jets_pt,jets_eta,jets_phi,jets_id,jets_CHF,90.,50.,800,3.5, false)'
pass_cuts_for_detajj = 'pass_VBF_cuts("detajj",jets_pt,jets_eta,jets_phi,jets_id,jets_CHF,90.,50.,800,3.5, false)'
pass_cuts_for_MET = 'mjj2>800. && pass_VBF_cuts("mjj",jets_pt,jets_eta,jets_phi,jets_id,jets_CHF,90.,50.,80,3.5, false)'




df_den['mjj'] = df_mjj_all.Filter(pass_cuts_for_mjj)
df_den['detajj'] = df_mjj_all.Filter(pass_cuts_for_detajj)
df_den['MET'] = df_mjj_all.Filter(pass_cuts_for_MET)


#df_num = { 'mjj' : {}, 'detajj' : {}, 'MET' : {} }
#df_num['mjj']['new'] = df_den['mjj'].Filter('passVBFmet_2j || passVBFmet_3j')
#df_num['mjj']['old'] = df_den['mjj'].Filter('passVBFmet_old_2j || passVBFmet_old_3j')
#df_num['mjj']['L1']  = df_den['mjj'].Filter('passL1VBFmet')
#
#df_num['detajj']['new'] = df_den['detajj'].Filter('passVBFmet_2j || passVBFmet_3j')
#df_num['detajj']['old'] = df_den['detajj'].Filter('passVBFmet_old_2j || passVBFmet_old_3j')
#df_num['detajj']['L1']  = df_den['detajj'].Filter('passL1VBFmet')
#
#df_num['MET']['new'] = df_den['MET'].Filter('passVBFmet_2j || passVBFmet_3j')
#df_num['MET']['old'] = df_den['MET'].Filter('passVBFmet_old_2j || passVBFmet_old_3j')
#df_num['MET']['L1']  = df_den['MET'].Filter('passL1VBFmet')


df_num = { 'mjj' : {}, 'detajj' : {}, 'MET' : {} }
df_num['mjj']['new'] = df_den['mjj'].Filter('passVBFmu_2j || passVBFmu_3j')
df_num['mjj']['old'] = df_den['mjj'].Filter('passVBFmet_old_2j || passVBFmet_old_3j')
df_num['mjj']['L1']  = df_den['mjj'].Filter('passL1VBFmu')

df_num['detajj']['new'] = df_den['detajj'].Filter('passVBFmu_2j || passVBFmu_3j')
df_num['detajj']['old'] = df_den['detajj'].Filter('passVBFmet_old_2j || passVBFmet_old_3j')
df_num['detajj']['L1']  = df_den['detajj'].Filter('passL1VBFmu')

df_num['MET']['new'] = df_den['MET'].Filter('passVBFmu_2j || passVBFmu_3j')
df_num['MET']['old'] = df_den['MET'].Filter('passVBFmet_old_2j || passVBFmet_old_3j')
df_num['MET']['L1']  = df_den['MET'].Filter('passL1VBFmu')


h_den = {}
h_num = {'mjj' : {}, 'detajj' : {}, 'MET' : {} }

h_den['mjj'] = df_den['mjj'].Histo1D(    ('h_den','',   15,0,2500),'mjj2')
h_num['mjj'] = {
    'new': df_num['mjj']['new'].Histo1D(    ('h_num_new','',   15,0,2500),'mjj2'),
    'old': df_num['mjj']['old'].Histo1D(    ('h_num_old','',   15,0,2500),'mjj2'),
    'L1':  df_num['mjj']['L1'].Histo1D(     ('h_num_l1','',    15,0,2500),'mjj2'),
}


h_den['detajj'] = df_den['detajj'].Histo1D(    ('h_den','',   15,0,10),'detajj')
h_num['detajj'] = {
    'new': df_num['detajj']['new'].Histo1D(    ('h_num_new','',   15,0,10),'detajj'),
    'old': df_num['detajj']['old'].Histo1D(    ('h_num_old','',   15,0,10),'detajj'),
    'L1':  df_num['detajj']['L1'].Histo1D(     ('h_num_l1','',    15,0,10),'detajj'),
}


#h_den['MET'] = df_den['MET'].Histo1D(    ('h_den','',   15,0,400),'METnoMu')
#h_num['MET'] = {
#    'new': df_num['MET']['new'].Histo1D(    ('h_num_new','',   15,0,400),'METnoMu'),
#    'old': df_num['MET']['old'].Histo1D(    ('h_num_old','',   15,0,400),'METnoMu'),
#    'L1':  df_num['MET']['L1'].Histo1D(     ('h_num_l1','',    15,0,400),'METnoMu'),
#}


h_den['MET'] = df_den['MET'].Histo1D(    ('h_den','',   15,0,400),'METnoMu')
h_num['MET'] = {
    'new': df_num['MET']['new'].Histo1D(    ('h_num_new','',   15,0,400),'METnoMu'),
    'old': df_num['MET']['old'].Histo1D(    ('h_num_old','',   15,0,400),'METnoMu'),
    'L1':  df_num['MET']['L1'].Histo1D(     ('h_num_l1','',    15,0,400),'METnoMu'),
}

Eff = {'mjj' : {}, 'detajj' : {}, 'MET' : {} }

Eff['mjj']['new'] = ROOT.TGraphAsymmErrors(h_num['mjj']['new'].GetValue(),  h_den['mjj'].GetValue(), 'eff_mjj')
Eff['mjj']['old'] = ROOT.TGraphAsymmErrors(h_num['mjj']['old'].GetValue(),  h_den['mjj'].GetValue(), 'eff_mjj')
Eff['mjj']['L1']  = ROOT.TGraphAsymmErrors(h_num['mjj']['L1'].GetValue(),   h_den['mjj'].GetValue(), 'eff_mjj')

Eff['detajj']['new'] = ROOT.TGraphAsymmErrors(h_num['detajj']['new'].GetValue(),  h_den['detajj'].GetValue(), 'eff_detajj')
Eff['detajj']['old'] = ROOT.TGraphAsymmErrors(h_num['detajj']['old'].GetValue(),  h_den['detajj'].GetValue(), 'eff_detajj')
Eff['detajj']['L1']  = ROOT.TGraphAsymmErrors(h_num['detajj']['L1'].GetValue(),   h_den['detajj'].GetValue(), 'eff_detajj')

Eff['MET']['new'] = ROOT.TGraphAsymmErrors(h_num['MET']['new'].GetValue(),  h_den['MET'].GetValue(), 'eff_MET')
Eff['MET']['old'] = ROOT.TGraphAsymmErrors(h_num['MET']['old'].GetValue(),  h_den['MET'].GetValue(), 'eff_MET')
Eff['MET']['L1']  = ROOT.TGraphAsymmErrors(h_num['MET']['L1'].GetValue(),   h_den['MET'].GetValue(), 'eff_MET')

In [ ]:
# TESTING PLOTS: VBF inclusive - L1 vs HLT efficiency
#h_den   = df_mjj_den.Histo1D(    ('h_den','',   15,0,4000),'mjj2')
#h_num   = df_mjj_num.Histo1D(    ('h_num','',   15,0,4000),'mjj2')
#h_numL1 = df_mjj_numL1.Histo1D(  ('h_numL1','', 15,0,4000),'mjj2')
#h_den   = df_mjj_den.Histo1D(    ('h_den','',   15,0,400),'METnoMu')
#h_num   = df_mjj_num.Histo1D(    ('h_num','',   15,0,400),'METnoMu')
#h_numL1 = df_mjj_numL1.Histo1D(  ('h_numL1','', 15,0,400),'METnoMu')
h_den   = df_mjj_den.Histo1D(    ('h_den','',   15,0,10),'Detajj')
h_num   = df_mjj_num.Histo1D(    ('h_num','',   15,0,10),'Detajj')
h_numL1 = df_mjj_numL1.Histo1D(  ('h_numL1','', 15,0,10),'Detajj')

print("den integral    = {}".format(h_den   .Integral()))
print("num integral    = {}".format(h_num   .Integral()))
print("numL1 integral  = {}".format(h_numL1 .Integral()))

Eff   = ROOT.TGraphAsymmErrors(h_num.GetValue(),  h_den.GetValue(), 'eff_mjj')
EffL1 = ROOT.TGraphAsymmErrors(h_numL1.GetValue(),h_den.GetValue(), 'eff_mjj')

c = ROOT.TCanvas()
c.SetGridx()
c.SetGridy()
c.SetTickx()
c.SetTicky()
Eff.GetYaxis().SetRangeUser(0.,1.3)
EffL1.GetYaxis().SetRangeUser(0.,1.3)
EffL1.GetYaxis().SetTitle('Trigger efficiency')
EffL1.GetXaxis().SetTitle('m_{jj} [GeV]')
EffL1.Draw('ap')
Eff.SetLineColor(2)
Eff.Draw('psame')
legend = ROOT.TLegend(0.5,0.15,0.9,0.5)
legend.SetFillStyle(0)
legend.SetBorderSize(0)
legend.AddEntry(EffL1,'VBFincl - L1','lp')
legend.AddEntry(Eff,  'VBFincl - HLT','lp')
legend.Draw("same")

c.Draw()

In [ ]:
# Definingvariables, categories & cuts

jet_vars = ['pt1','pt2','mjj','detajj']

trigs = ['VBFincl','VBFjets','VBFmet' ,'VBFphot','VBFele' ,'VBFmu'  ,'VBFtau']

jet_cuts = {
        'VBFincl' : {"pt1": 115, "pt2":50, "mjj":1200, "detajj":3.8},
        'VBFjets' : {"pt1": 80,  "pt2":50, "mjj":800,  "detajj":2.8},
        'VBFmet'  : {"pt1": 115,  "pt2":50, "mjj":800,  "detajj":2.8},
        'VBFphot' : {"pt1": 55,  "pt2":55, "mjj":700,  "detajj":2.8},
        'VBFele'  : {"pt1": 55,  "pt2":55, "mjj":700,  "detajj":2.8},
        'VBFmu'   : {"pt1": 100, "pt2":50, "mjj":800,  "detajj":2.8},
        'VBFtau'  : {"pt1": 55,  "pt2":55, "mjj":700,  "detajj":2.8},
}

cut_str = {
    'mjj'    : 'detajj > {detajjcut} && pt2 > {pt2cut} && ((pt1>{pt1cut}) || (pt1>{pt2cut}&&j1_pt>{pt1cut}))',
    'pt1'    : 'detajj > {detajjcut} && pt2 > {pt2cut} && mjj > {mjjcut}',
    'pt2'    : 'detajj > {detajjcut} && mjj > {mjjcut} && ((pt1>{pt1cut}) || (pt1>{pt2cut}&&j1_pt>{pt1cut}))',
    'detajj' : 'mjj > {mjjcut} && pt2 > {pt2cut} && ((pt1>{pt1cut}) || (pt1>{pt2cut}&&j1_pt>{pt1cut}))',
    'centr'  : 'detajj > {detajjcut} && mjj > {mjjcut} && pt2 > {pt2cut} && ((pt1>{pt1cut}) || (pt1>{pt2cut}&&j1_pt>{pt1cut}))',
}

add_cut_str = {
        'VBFincl' : '1==1',
        'VBFjets' : 'passSelec_VBFjets==1',
        'VBFmet'  : 'MET_et>170',
        'VBFphot' : 'passSelec_VBFphot==1',
        'VBFele'  : '0==1',
        'VBFmu'   : '1==1',
        'VBFtau'  : '0==1',
}

add_cut_label = {
    #'mjj'    : '#Delta#eta(j1,j2)>3.5, MET(no-#mu) > 200 GeV',
    'mjj'    : '#Delta#eta(j1,j2)>3.5',
    #'detajj' : 'm_{jj} > 800 GeV,MET(no-#mu) > 200 GeV',
    'detajj' : 'm_{jj} > 800 GeV',
    'MET' : 'm_{jj} > 800 GeV, #Delta#eta(j1,j2)>3.5',
}

# Plotting labels
VARLABELS = {
    'mjj'    : 'm_{jj} [GeV]',
    'pt1'    : 'Leading jet p_{T} [GeV]',
    'pt2'    : 'Subleading jet p_{T} [GeV]',
    'detajj' : '#Delta#eta(j1,j2)',
    'MET'    : 'MET (no-#mu) [GeV]'
}


TRIGLABEL = {
    'VBFincl' : "HLT_VBF_DiPFJet105_40_Mjj1000_Detajj3p5*",
    'VBFjets' : "HLT_VBF_DiPFJet70_40_Mjj600_Detajj2p5_DiPFJet60*",
    'VBFmet'  : "HLT_VBF_DiPFJet75_40_Mjj500_Detajj2p5_PFMET85*",
    'VBFphot' : "HLT_VBF_DiPFJet45_Mjj500_Detajj2p5_Photon12*",
    'VBFele'  : "HLT_VBF_DiPFJet45_Mjj500_Detajj2p5_Ele12_eta2p1_WPTight_Gsf*",
    'VBFmu'   : "HLT_VBF_DiPFJet90_40_Mjj600_Detajj2p5_Mu3_TrkIsoVVL*",
    'VBFtau'  : "HLT_VBF_DiPFJet45_Mjj500_Detajj2p5_MediumDeepTauPFTauHPS45*",
}

import CMS_lumi, tdrstyle
import array

def redrawBorder():
    ROOT.gPad.Update()
    ROOT.gPad.RedrawAxis()
    l = ROOT.TLine()
    l.SetLineWidth(2)
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymax(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymin())

print(CMS_lumi.__file__)

tdrstyle.setTDRStyle()

#change the CMS_lumi variables (see CMS_lumi.py)
CMS_lumi.lumi_7TeV = "4.8 fb^{-1}"
CMS_lumi.lumi_8TeV = "18.3 fb^{-1}"
CMS_lumi.writeExtraText = True
CMS_lumi.extraText = "Preliminary"
CMS_lumi.lumi_sqrtS = "2023 C (13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)
iPeriod = 0

iPos = 0#11

H_ref = 600; 
W_ref = 600; 
W = W_ref
H = H_ref

# references for T, B, L, R
T = 0.08*H_ref
B = 0.14*H_ref 
L = 0.12*W_ref
R = 0.04*W_ref

theVar = 'mjj'
theTrig = 'VBFmet'

path = {}
legend = {}
canvas = {}
hpx = {}

text_coord = {
    'mjj' :    [1000,0.1,2400,0.7],
    'detajj' : [3.3,0.1,7.85,0.7],
    'MET' :    [170,0.1,390,0.7]
    
}

In [ ]:

for theVar in ['mjj','detajj','MET']:

    Eff[theVar]['L1'].SetLineColor(8)
    Eff[theVar]['L1'].SetMarkerColor(8)
    Eff[theVar]['L1'].SetLineWidth(2)
    Eff[theVar]['L1'].SetMarkerStyle(22)
    
    
    Eff[theVar]['new'].SetLineColor(46)
    Eff[theVar]['new'].SetMarkerColor(46)
    Eff[theVar]['new'].SetLineWidth(2)
    Eff[theVar]['new'].SetMarkerStyle(23)
    
    
    Eff[theVar]['old'].SetLineColor(9)
    Eff[theVar]['old'].SetMarkerColor(9)
    Eff[theVar]['old'].SetLineWidth(2)
    Eff[theVar]['old'].SetMarkerStyle(20)    
    
    
    path[theVar] = ROOT.TPaveText(text_coord[theVar][0],
                                  text_coord[theVar][1],
                                  text_coord[theVar][2],
                                  text_coord[theVar][3])
    #path[theVar] = ROOT.TPaveText(40,0.8,110,1.2)
    
    path[theVar].SetTextAlign(12)
    path[theVar].SetFillStyle(0)
    path[theVar].SetBorderSize(0)
    #        'Selection: 
    # #newline #geq 1 ak4 jets (tight ID), 
    # #newline p_{T1}>140 GeV, p_{T2}>70 GeV, 
    # #newline #Delta#eta(j1,j2)>3.5 #
    # #newline Ref.: IsoMu27, > 1 #mu (tight ID, pT > 30 GeV)',  
    
#    path[theVar].AddText("Path: {thepath}".format(thepath=TRIGLABEL['VBFmet
#    path[theVar].AddText("Path: {thepath}".format(thepath=TRIGLABEL['VBFincl']))
    path[theVar].AddText("Path: {thepath}".format(thepath=TRIGLABEL['VBFmu']))
    path[theVar].AddText('Offline selection:')
    path[theVar].AddText('> 1 ak4 jets (tight ID), ')
    #path[theVar].AddText('p_{T1}>125 GeV, p_{T2}>50 GeV, ') 
    path[theVar].AddText('p_{T1}>125 GeV, p_{T2}>50 GeV, ') 
    path[theVar].AddText(add_cut_label[theVar])
    path[theVar].AddText('Ref.: IsoMu27, #geq 1#mu (tight ID, pT > 30 GeV) ')
    
    #path.AddText('#geq 1 b-tagged large radius (R=0.8) jets,')
    #path.AddText('p_{T}>250 GeV, m_{SD} > 40 GeV')  
    
    
    
    legend[theVar] = ROOT.TLegend(0.65,0.75,0.90,0.90)
    legend[theVar].SetFillStyle(0)
    legend[theVar].SetBorderSize(0
    legend[theVar].AddEntry(Eff[theVar]['L1'],   'VBF+mu - L1','lp')
    legend[theVar].AddEntry(Eff[theVar]['new'],  'VBF+mu - HLT','lp')
#    legend[theVar].AddEntry(Eff[theVar]['L1'],   'VBF inclusive - L1','lp')
#    legend[theVar].AddEntry(Eff[theVar]['new'],  'VBF inclusive - HLT','lp')
    #legend[theVar].AddEntry(Eff[theVar]['L1'],   'VBF+MET - L1','lp')
    #legend[theVar].AddEntry(Eff[theVar]['new'],  'VBF+MET - HLT','lp')
#    legend[theVar].AddEntry(Eff[theVar]['old'],  'VBF+MET - old HLT','lp')
    
    
    canvas[theVar]=ROOT.TCanvas(theVar,"Trigger Efficicency")
    canvas[theVar].SetFillColor(0)
    canvas[theVar].SetBorderMode(0)
    canvas[theVar].SetFrameFillStyle(0)
    canvas[theVar].SetFrameBorderMode(0)
    canvas[theVar].SetLeftMargin( L/W )
    canvas[theVar].SetRightMargin( R/W )
    canvas[theVar].SetTopMargin( T/H )
    canvas[theVar].SetBottomMargin( B/H )
    canvas[theVar].SetTickx()
    canvas[theVar].SetTicky()
    canvas[theVar].SetGrid()
    
    #canvas.Update()
    #canvas.RedrawAxis()
    
    #anvas.SetGrid()
    canvas[theVar].cd()
    
    #dummy hist for consistent display
    
    xlow=Eff[theVar]['L1'].GetXaxis().GetBinLowEdge(0)
    xhigh=Eff[theVar]['L1'].GetXaxis().GetBinUpEdge(92)
    hpx[theVar] = ROOT.TH2F("hpx","",10,xlow,xhigh,10,-0.02,1.3);
    hpx[theVar].SetStats(False)
    
    hpx[theVar].SetTitle(theVar)
    hpx[theVar].GetXaxis().SetTitle(VARLABELS[theVar])
    hpx[theVar].GetYaxis().SetTitle("Trigger Efficiency")
    hpx[theVar].GetXaxis().SetTitleSize(0.055)
    hpx[theVar].GetXaxis().SetTitleOffset(1.1)
    hpx[theVar].GetYaxis().SetTitleSize(0.06)
    hpx[theVar].GetYaxis().SetTitleOffset(0.9)
    hpx[theVar].Draw()
        
    Eff[theVar]['L1'].Draw("p same")
    #Eff[theVar]['old'].Draw("p same")
    Eff[theVar]['new'].Draw("p same")
    legend[theVar].Draw("same")
    path[theVar].Draw("same")
    canvas[theVar].Update()
    CMS_lumi.CMS_lumi(canvas[theVar], iPeriod, iPos)
    redrawBorder()
    canvas[theVar].Draw()

In [ ]:
# Fill one DF per (variable to plot)x(trigger to check)
# only for VBF-jet variables for now (TODO: central objects)

dfs = {}
for trig in trigs:
    dfs[trig] = {}
    dfs[trig]['mjj'] = df.Filter(add_cut_str[trig]).Filter(cut_str['mjj'].format(detajjcut=jet_cuts[trig]['detajj'],
                                                pt1cut=jet_cuts[trig]['pt1'],
                                                pt2cut=jet_cuts[trig]['pt2']))
    
    dfs[trig]['pt1'] = df.Filter(add_cut_str[trig]).Filter(cut_str['pt1'].format(detajjcut=jet_cuts[trig]['detajj'],
                                                pt2cut=jet_cuts[trig]['pt2'],
                                                mjjcut=jet_cuts[trig]['mjj']))
    
    dfs[trig]['pt2'] = df.Filter(add_cut_str[trig]).Filter(cut_str['pt2'].format(detajjcut=jet_cuts[trig]['detajj'],
                                                pt1cut=jet_cuts[trig]['pt1'],
                                                pt2cut=jet_cuts[trig]['pt2'],
                                                mjjcut=jet_cuts[trig]['mjj']))
    
    dfs[trig]['detajj'] = df.Filter(add_cut_str[trig]).Filter(cut_str['detajj'].format(mjjcut=jet_cuts[trig]['mjj'],
                                                             pt1cut=jet_cuts[trig]['pt1'],
                                                             pt2cut=jet_cuts[trig]['pt2']))
    
    

In [ ]:
h_den = {}
h_num = {}
Eff = {}
for trig in trigs:
    h_den[trig] = {}
    h_num[trig] = {}
    Eff[trig] = {}
    for var in jet_vars:
        h_num[trig][var] = dfs[trig][var].Filter('pass{trg}==1 && passSelec_{trg}==1'.format(trg=trig)).Histo1D(binning[var][0],var)
        h_den[trig][var] = dfs[trig][var].Filter('passSelec_{trg}==1'.format(trg=trig)).Histo1D(binning[var][1],var)
        Eff[trig][var] = ROOT.TGraphAsymmErrors(h_num[trig][var].GetValue(),h_den[trig][var].GetValue(),'eff_{}_{}'.format(trig,var))

In [ ]:
def redrawBorder():
    ROOT.gPad.Update()
    ROOT.gPad.RedrawAxis()
    l = ROOT.TLine()
    l.SetLineWidth(2)
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymax(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymin())

In [ ]:
import CMS_lumi, tdrstyle
import array

print(CMS_lumi.__file__)

tdrstyle.setTDRStyle()

#change the CMS_lumi variables (see CMS_lumi.py)
CMS_lumi.lumi_7TeV = "4.8 fb^{-1}"
CMS_lumi.lumi_8TeV = "18.3 fb^{-1}"
CMS_lumi.writeExtraText = True
CMS_lumi.extraText = "Preliminary"
CMS_lumi.lumi_sqrtS = "2023 C (13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)
iPeriod = 0

iPos = 0#11

H_ref = 600; 
W_ref = 800; 
W = W_ref
H  = H_ref

# references for T, B, L, R
T = 0.08*H_ref
B = 0.14*H_ref 
L = 0.12*W_ref
R = 0.04*W_ref


# Draw one plot for test
leg_x0 = 0.51
leg_y0 = 0.15
leg_x1 = 0.89
leg_y1 = 0.5

leg1_x0 = 0.15
leg1_y0 = 0.55
leg1_x1 = 0.49
leg1_y1 = 0.89

leg2_x0 = 0.15
leg2_y0 = 0.15
leg2_x1 = 0.49
leg2_y1 = 0.45

theVar = 'mjj'
theTrig = 'VBFmet'

path = {}
legend = {}
canvas = {}
hpx = {}
for theTrig in trigs:

    Eff[theTrig][theVar].SetLineColor(8)
    Eff[theTrig][theVar].SetMarkerColor(8)
    Eff[theTrig][theVar].SetLineWidth(2)
    Eff[theTrig][theVar].SetMarkerStyle(22)
    
    
    
    
    path[theTrig] = ROOT.TPaveText(30,0.8,1250,1.2)
    #path[theTrig] = ROOT.TPaveText(40,0.8,110,1.2)
    
    path[theTrig].SetTextAlign(12)
    path[theTrig].SetFillStyle(0)
    path[theTrig].SetBorderSize(0)
    #        'Selection: 
    # #newline #geq 1 ak4 jets (tight ID), 
    # #newline p_{T1}>140 GeV, p_{T2}>70 GeV, 
    # #newline #Delta#eta(j1,j2)>3.5 #
    # #newline Ref.: IsoMu27, > 1 #mu (tight ID, pT > 30 GeV)',  
    
    path[theTrig].AddText("Path: {thepath}".format(thepath=TRIGLABEL[theTrig]))
    path[theTrig].AddText('Offline selection:')
    path[theTrig].AddText('> 1 ak4 jets (tight ID), ')
    path[theTrig].AddText('p_{T1}>140 GeV, p_{T2}>70 GeV, ') 
    #path[theTrig].AddText('p_{T2}>70 GeV, m_{jj}>1200 GeV') 
    path[theTrig].AddText('#Delta#eta(j1,j2)>3.5 ')
    if theTrig in ['VBFjets','VBFmet','VBFphot']:
        path[theTrig].AddText(add_cut_label[theTrig])
    path[theTrig].AddText('Ref.: IsoMu27, #geq 1#mu (tight ID, pT > 30 GeV) ')
    
    #path.AddText('#geq 1 b-tagged large radius (R=0.8) jets,')
    #path.AddText('p_{T}>250 GeV, m_{SD} > 40 GeV')  
    
    
    
    
    #print(TRIG_INFO[theTrig]['path'])
    #path.SetFontSize(7)
    legend[theTrig] = ROOT.TLegend(0.16,0.65,0.4,0.9)
    legend[theTrig].SetFillStyle(0)
    legend[theTrig].SetBorderSize(0)
    canvas[theTrig]=ROOT.TCanvas(theTrig,"Trigger Efficicency")
    
    canvas[theTrig].SetFillColor(0)
    canvas[theTrig].SetBorderMode(0)
    canvas[theTrig].SetFrameFillStyle(0)
    canvas[theTrig].SetFrameBorderMode(0)
    canvas[theTrig].SetLeftMargin( L/W )
    canvas[theTrig].SetRightMargin( R/W )
    canvas[theTrig].SetTopMargin( T/H )
    canvas[theTrig].SetBottomMargin( B/H )
    canvas[theTrig].SetTickx()
    canvas[theTrig].SetTicky()
    canvas[theTrig].SetGrid()
    
    #canvas.Update()
    #canvas.RedrawAxis()
    
    #anvas.SetGrid()
    canvas[theTrig].cd()
    
    #dummy hist for consistent display
    
    xlow=Eff[theTrig][theVar].GetXaxis().GetBinLowEdge(0)
    xhigh=Eff[theTrig][theVar].GetXaxis().GetBinUpEdge(92)
    hpx[theTrig] = ROOT.TH2F("hpx","",10,xlow,xhigh,10,-0.02,1.3);
    hpx[theTrig].SetStats(False)
    
    hpx[theTrig].SetTitle(theTrig)
    hpx[theTrig].GetXaxis().SetTitle(VARLABELS[theVar])
    hpx[theTrig].GetYaxis().SetTitle("Trigger Efficiency")
    hpx[theTrig].GetXaxis().SetTitleSize(0.055)
    hpx[theTrig].GetXaxis().SetTitleOffset(1.1)
    hpx[theTrig].GetYaxis().SetTitleSize(0.06)
    hpx[theTrig].GetYaxis().SetTitleOffset(0.9)
    hpx[theTrig].Draw()
        
    Eff[theTrig][theVar].Draw("p same")
    legend[theTrig].Draw("same")
    path[theTrig].Draw("same")
    canvas[theTrig].Update()
    CMS_lumi.CMS_lumi(canvas[theTrig], iPeriod, iPos)
    redrawBorder()
    canvas[theTrig].Draw()